# Evergreen Family Income: backtest notebook

**Group BE** · Nova SBE 2270 Financial Modeling · *Dividends to live on. Capital to pass on.*

This notebook implements the strategy paper *Evergreen Family Income: Strategy & Allocation Rule* as written and reproduces every number, table and figure used in the prospectus.

| Sleeve | Weight | Instrument | Role |
|---|---|---|---|
| Eurozone dividend stocks | 30% | 20 stocks picked by our rule, equal weight | today's income in EUR |
| US dividend stocks | 30% | SPDR S&P U.S. Dividend Aristocrats UCITS (SPYD), unhedged | income in USD, diversifier |
| Euro government bonds | 20% | iShares Core € Govt Bond (EUNH) | contractual income, crisis ballast |
| Euro corporate bonds | 15% | iShares Core € Corp Bond (EUN5) | extra contractual income |
| Gold | 5% | Xetra-Gold (4GLD) | crisis insurance |

**Setup.** Open in Google Colab → *Runtime → Run all*. All data comes through APIs (Yahoo Finance via `yfinance`, FRED); no data files are needed. The first run takes about 5 minutes. Start and end dates are set explicitly in section 2.

**Contents.** 1 Core functions · 2 Parameters · 3 Planning assumptions · 4 Universe and data · 5 Data checks · 6 Stock selection · 7 Backtest vs. benchmarks · 8 Sleeve attribution · 9 Client view · 10 Stress years · 11 Sensitivity · 12 Export · 13 Limitations

## 1. Core functions
This cell writes `evergreen_core.py`. The notebook **and** the Streamlit app import the same functions, so both compute identical numbers.

In [ ]:
%%writefile evergreen_core.py
"""
evergreen_core.py - core functions of the Evergreen Family Income strategy.
Imported by the notebook AND by the Streamlit app, so both compute identical numbers.

Sections: 1) data  2) Eurozone stock selection  3) simulation  4) measures  5) planning tables
"""
import time
import numpy as np
import pandas as pd


# =====================================================================
# 1) Data
# =====================================================================
def download_prices(tickers, start, end, chunk=40, retries=3, pause=5):
    """Month-end closes (split-adjusted, NOT dividend-adjusted) and dividends per share per month
    (sum of ex-dates in the month) from Yahoo Finance. Never use Adj Close for yields."""
    import yfinance as yf
    end_excl = (pd.Period(end, 'M') + 1).to_timestamp().strftime('%Y-%m-%d')
    closes, divs = [], []
    for i in range(0, len(tickers), chunk):
        part, df = list(tickers[i:i + chunk]), None
        for _ in range(retries):
            try:
                df = yf.download(part, start=start, end=end_excl, interval='1d', auto_adjust=False,
                                 actions=True, progress=False, threads=True)
                if df is not None and not df.empty:
                    break
            except Exception as e:
                print('retry:', e)
            time.sleep(pause)
        if df is None or df.empty:
            continue
        c = df['Close']
        d = df['Dividends'] if 'Dividends' in df.columns.get_level_values(0) else c * 0
        if isinstance(c, pd.Series):
            c, d = c.to_frame(part[0]), d.to_frame(part[0])
        closes.append(c)
        divs.append(d)
    close = pd.concat(closes, axis=1)
    close = close.loc[:, ~close.columns.duplicated()]
    div = pd.concat(divs, axis=1)
    div = div.loc[:, ~div.columns.duplicated()].reindex(close.index).fillna(0.0)
    close_m = close.resample('ME').last()
    div_m = div.resample('ME').sum()
    for x in (close_m, div_m):
        x.index = x.index.to_period('M')
    keep = close_m.columns[close_m.notna().any()]
    return close_m[keep], div_m[keep]


def load_fred(series_id):
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}'
    s = pd.read_csv(url, index_col=0, parse_dates=True).iloc[:, 0]
    s = pd.to_numeric(s, errors='coerce').dropna()
    s.index = s.index.to_period('M')
    return s


def usd_to_eur(close_usd, div_usd, eurusd):
    """Convert a USD-listed series to EUR with month-end EUR/USD (USD per 1 EUR)."""
    fx = eurusd.reindex(close_usd.index).ffill()
    return close_usd / fx, div_usd / fx


def fx_hedged_series(close_eur, div_eur, eurusd, r_eur_ann, r_usd_ann, cost_bps=10):
    """Synthetic EUR-hedged version of a USD asset quoted in EUR (app / sensitivity option).
    hedged return ~ (1 + USD local return) * (1 + (i_EUR - i_USD)/12 - cost/12) - 1  (covered interest parity).
    Distributions keep the unhedged dividend yield, applied to the hedged price."""
    idx = close_eur.index
    p_usd = close_eur * eurusd.reindex(idx).ffill()
    r_loc = p_usd / p_usd.shift(1) - 1
    carry = (r_eur_ann.reindex(idx).ffill() - r_usd_ann.reindex(idx).ffill()) / 12 - cost_bps / 1e4 / 12
    gross = ((1 + r_loc) * (1 + carry.fillna(0))).fillna(1.0)
    first = close_eur.first_valid_index()
    g = gross.loc[first:].copy()
    g.iloc[0] = 1.0
    hp = (g.cumprod() * close_eur.loc[first]).reindex(idx)
    hd = (div_eur / close_eur).fillna(0) * hp
    return hp, hd.fillna(0)


def total_return(px, div):
    """Monthly total return of a single instrument from close + dividends."""
    return (px + div) / px.shift(1) - 1


# =====================================================================
# 2) Eurozone stock selection - uses ONLY data up to and including t
# =====================================================================
FTT = {   # financial transaction tax on share PURCHASES: list of (first month, rate)
    'France': [('2012-08', 0.002), ('2017-01', 0.003), ('2025-04', 0.004)],
    'Italy':  [('2013-03', 0.0012), ('2014-01', 0.001), ('2026-01', 0.002)],
    'Spain':  [('2021-01', 0.002)],
}


def ftt_rate(country, t):
    r = 0.0
    for start, rate in FTT.get(country, []):
        if t >= pd.Period(start, 'M'):
            r = rate
    return r


def dividend_screen(annual, last_year, stocks, R):
    """Dividend-record screen on calendar-year DPS (ex-date year).
    R['hist_years']    : length of the window (5)
    R['min_paid_years']: years in the window with DPS > 0 (5 = every year)
    R['cut_tol']       : max. year-on-year cut (0.10); None = no year-on-year test
    R['net_growth']    : DPS in the latest year >= DPS in the first year of the window"""
    yrs = list(range(last_year - R['hist_years'] + 1, last_year + 1))
    a = annual.reindex(index=yrs, columns=stocks).fillna(0.0)
    ok = ((a > 0).sum() >= R['min_paid_years']) & (a.iloc[-1] > 0)
    if R.get('cut_tol') is not None:
        prev, cur = a.iloc[:-1].values, a.iloc[1:].values
        with np.errstate(divide='ignore', invalid='ignore'):
            ratio = np.where(prev > 0, cur / prev, np.where(cur > 0, np.inf, 0.0))
        ok &= pd.Series((ratio >= 1 - R['cut_tol']).all(axis=0), index=stocks)
    if R.get('net_growth', True):
        ok &= a.iloc[-1] >= a.iloc[0] * (1 - 1e-9)
    return ok


def select_stocks(t, close_raw, div_m, meta, R, incumbents=()):
    """Eurozone selection on date t. Returns
    w      : weights within the Eurozone sleeve (sum 1; empty slots go to R['fallback'])
    info   : table of the selected stocks
    counts : how many stocks pass each screen (data check / prospectus)"""
    close_h, div_h = close_raw.loc[:t], div_m.loc[:t]          # nothing after t is visible
    assert close_h.index.max() <= t and div_h.index.max() <= t, 'look-ahead'
    stocks = meta.index.intersection(close_h.columns)
    p_t, p_12 = close_h.loc[t, stocks], close_h.loc[t - 12, stocks]
    listed = p_t.notna() & p_12.notna() & (p_t > 0)
    u = listed[listed].index

    annual = div_h[u].groupby(div_h.index.year).sum()
    last_year = t.year if t.month == 12 else t.year - 1
    record = dividend_screen(annual, last_year, u, R)
    y = div_h.loc[t - 11:t, u].sum() / p_t[u]                   # trailing 12-month yield
    r12 = p_t[u] / p_12[u] - 1                                  # 12-month price return
    loser = (r12 <= r12.quantile(R['loser_quantile'])) & (r12 < 0)
    yield_ok = (y > 0) & (y <= R['max_yield'])

    surv = y[record & yield_ok & ~loser].sort_values(ascending=False)
    rank = pd.Series(np.arange(1, len(surv) + 1), index=surv.index)
    inc = set(incumbents)
    keep = [s for s in surv.index if s in inc and rank[s] <= R['buffer_rank']]
    order = keep + [s for s in surv.index if s not in set(keep)]
    picked, n_sec, n_cty = [], {}, {}
    for s in order:
        sec, cty = meta.at[s, 'sector'], meta.at[s, 'country']
        if n_sec.get(sec, 0) >= R['max_per_sector'] or n_cty.get(cty, 0) >= R['max_per_country']:
            continue
        picked.append(s)
        n_sec[sec] = n_sec.get(sec, 0) + 1
        n_cty[cty] = n_cty.get(cty, 0) + 1
        if len(picked) == R['n_stocks']:
            break

    w = pd.Series(1.0 / R['n_stocks'], index=picked, dtype=float)
    if len(picked) < R['n_stocks']:
        w[R['fallback']] = (R['n_stocks'] - len(picked)) / R['n_stocks']
    info = pd.DataFrame({'name': meta.loc[picked, 'name'], 'sector': meta.loc[picked, 'sector'],
                         'country': meta.loc[picked, 'country'], 'yield_ttm': y.reindex(picked),
                         'rank': rank.reindex(picked), 'kept_by_buffer': [s in keep for s in picked]})
    counts = {'universe': len(stocks), 'listed_12m': len(u), 'dividend_record': int(record.sum()),
              'yield_in_range': int(yield_ok.sum()), 'losers_dropped': int(loser.sum()),
              'eligible': len(surv), 'selected': len(picked), 'fallback_slots': R['n_stocks'] - len(picked)}
    return w, info, counts


def run_selection(dates, close_raw, div_m, meta, R):
    """Sequential yearly selection (the buffer needs last year's holdings)."""
    out, inc = {}, []
    for t in dates:
        w, info, cnt = select_stocks(t, close_raw, div_m, meta, R, inc)
        out[t] = (w, info, cnt)
        inc = list(info.index)
    return out


def equal_weight_universe(t, close_raw, meta):
    """1/N diagnostic: every universe stock with a 12-month price history at t."""
    c = close_raw.loc[:t]
    stocks = meta.index.intersection(c.columns)
    ok = c.loc[t, stocks].notna() & c.loc[t - 12, stocks].notna()
    sel = ok[ok].index
    return pd.Series(1 / len(sel), index=sel)


# =====================================================================
# 3) Simulation in share counts
# =====================================================================
def simulate(spec, px, div, start, end, cfg, last_valid=None):
    """
    spec : {sleeve: {'weight': w, 'group': 'EQ'|'BOND'|'GOLD', 'fn': f(t) -> weights within the sleeve,
                     'fallback': ticker for delisted holdings (optional)}}
    px   : month-end prices, forward-filled;  div: dividends per share in the month (ex-date)
    cfg  : costs, payout rule, rebalancing (see notebook, section 2)
    Each month: dividends/coupons arrive -> payout -> (December) reselect + reset all sleeves,
    (cfg['band_months']) reset all sleeves if a group leaves its band, keeping current weights inside sleeves.
    ret = (value at month end + payout) / value at previous month end - 1  (time-weighted, net of costs).
    """
    months = pd.period_range(start, end, freq='M')
    W = pd.Series({s: v['weight'] for s, v in spec.items() if v['weight'] > 0}, dtype=float)
    assert abs(W.sum() - 1) < 1e-9, f'sleeve weights sum to {W.sum()}'
    grp = pd.Series({s: spec[s]['group'] for s in W.index})
    tgt_g = W.groupby(grp).sum()
    etfs, country = cfg['etfs'], cfg['country']
    cm = cfg.get('cost_mult', 1.0)
    shares = {}

    def holdings_value(p):
        return pd.Series({s: float((sh * p.reindex(sh.index).fillna(0)).sum()) for s, sh in shares.items()},
                         dtype=float).reindex(W.index).fillna(0.0)

    def by_ticker(d):
        if not d:
            return pd.Series(dtype=float)
        return pd.concat(list(d.values())).groupby(level=0).sum()

    def trade_to(total, sleeve_w, within, p, t):
        old = by_ticker({s: sh * p.reindex(sh.index).fillna(0) for s, sh in shares.items()})
        new = by_ticker({s: total * sleeve_w[s] * within[s] for s in sleeve_w.index if sleeve_w[s] > 0})
        delta = new.subtract(old, fill_value=0.0)
        tc = np.array([cfg['tc_etf'] if k in etfs else cfg['tc_stock'] for k in delta.index])
        ftt = np.array([0.0 if k in etfs else ftt_rate(country.get(k, ''), t) for k in delta.index])
        cost = cm * float((delta.abs().values * tc).sum() + (delta.clip(lower=0).values * ftt).sum())
        net = total - cost
        shares.clear()
        for s in sleeve_w.index:
            if sleeve_w[s] <= 0:
                continue
            ww = within[s]
            shares[s] = (net * sleeve_w[s] * ww / p.reindex(ww.index)).replace([np.inf, -np.inf], 0).fillna(0)
        return cost

    def current_within(p):
        out = {}
        for s, sh in shares.items():
            v = sh * p.reindex(sh.index).fillna(0)
            out[s] = v / v.sum() if v.sum() > 0 else v
        return out

    t0 = months[0]
    p = px.loc[t0]
    cost = trade_to(cfg['w0'], W, {s: spec[s]['fn'](t0) for s in W.index}, p, t0)
    prev = float(holdings_value(p).sum())
    hist_v = [prev]
    rows = [{'period': t0, 'value': prev, 'income': 0.0, 'payout': 0.0, 'cost': cost, 'ret': np.nan,
             'event': 'start', **{f'w_{g}': float(tgt_g[g]) for g in tgt_g.index}}]
    for m in months[1:]:
        p, d = px.loc[m], div.loc[m]
        # takeovers / delistings: proceeds go to the sleeve's fallback ETF until the next review
        if last_valid is not None:
            for s in list(shares):
                sh = shares[s]
                dead = [k for k in sh.index if sh[k] > 0 and k in last_valid and last_valid[k] is not None
                        and m > last_valid[k]]
                if dead:
                    fb = spec[s].get('fallback', cfg['fallback'])
                    val = float((sh[dead] * p.reindex(dead)).sum())
                    sh = sh.drop(dead)
                    sh[fb] = sh.get(fb, 0.0) + val / p[fb]
                    shares[s] = sh
        income = float(sum((sh * d.reindex(sh.index).fillna(0)).sum() for sh in shares.values()))
        vals = holdings_value(p)
        V = float(vals.sum())
        if cfg['payout'] == 'income':
            wd = income
        elif cfg['payout'] == 'smoothed':                       # rate x average value of the last N months
            wd = cfg['payout_rate'] / 12 * float(np.mean((hist_v + [V])[-cfg['smooth_months']:]))
        else:
            raise ValueError(cfg['payout'])
        wd = min(wd, V + income)
        cash = income - wd
        cost, event = 0.0, ''
        gw = (vals.groupby(grp).sum() / V) if V > 0 else tgt_g * 0
        if m.month == cfg['rebal_month']:
            cost = trade_to(V + cash, W, {s: spec[s]['fn'](m) for s in W.index}, p, m)
            event = 'annual'
        elif m.month in cfg['band_months'] and any(abs(gw[g] - tgt_g[g]) > cfg['bands'].get(g, 1.0)
                                                   for g in tgt_g.index):
            cost = trade_to(V + cash, W, current_within(p), p, m)
            event = 'band'
        elif abs(cash) > 1e-6:                                  # smoothed payout: buy / sell pro rata
            cost = trade_to(V + cash, vals / V, current_within(p), p, m)
            event = 'flow'
        end_v = float(holdings_value(p).sum())
        gw_end = holdings_value(p).groupby(grp).sum() / end_v
        rows.append({'period': m, 'value': end_v, 'income': income, 'payout': wd, 'cost': cost,
                     'ret': (end_v + wd) / prev - 1 if prev > 0 else np.nan, 'event': event,
                     **{f'w_{g}': float(gw_end[g]) for g in tgt_g.index}})
        prev = end_v
        hist_v.append(end_v)
    return pd.DataFrame(rows).set_index('period')


# =====================================================================
# 4) Performance measures (monthly data)
# =====================================================================
def standalone_stats(r, rf, periods=12):
    r = r.dropna()
    rf = rf.reindex(r.index).fillna(0)
    ex = r - rf
    c = (1 + r).cumprod()
    roll12 = (1 + r).rolling(12).apply(np.prod, raw=True) - 1
    return pd.Series({
        'Ann. average return': r.mean() * periods,
        'CAGR': c.iloc[-1] ** (periods / len(r)) - 1,
        'Volatility': r.std() * np.sqrt(periods),
        'Sharpe ratio': ex.mean() * periods / (ex.std() * np.sqrt(periods)),
        'Max drawdown': (c / c.cummax() - 1).min(),
        'Worst 12-month return': roll12.min(),
    })


def relative_stats(r, bench, rf, periods=12):
    """Beta = slope of strategy excess on benchmark excess returns (not annualised);
    alpha = 12 x intercept, with its OLS t-statistic; IR = 12 x mean active / (sqrt(12) x tracking error)."""
    idx = r.dropna().index.intersection(bench.dropna().index)
    r, b, rf = r.loc[idx], bench.loc[idx], rf.reindex(idx).fillna(0)
    y, x = (r - rf).values, (b - rf).values
    X = np.column_stack([np.ones_like(x), x])
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ coef
    s2 = resid @ resid / (len(y) - 2)
    se = np.sqrt(np.diag(s2 * np.linalg.inv(X.T @ X)))
    active = r - b
    te = active.std() * np.sqrt(periods)
    return pd.Series({
        'Beta': coef[1],
        'Alpha (ann.)': coef[0] * periods,
        't(alpha)': coef[0] / se[0],
        'Tracking error': te,
        'Information ratio': active.mean() * periods / te if te > 1e-12 else np.nan,
    })


def client_stats(sim, cpi, w0):
    """What the family gets: income in EUR, its stability, and real wealth after payouts."""
    s = sim.iloc[1:]
    yrs = s.index.year
    full = [y for y, n in pd.Series(yrs).value_counts().items() if n == 12]
    inc = s['income'].groupby(yrs).sum().loc[sorted(full)]
    start_val = sim['value'].shift(1).iloc[1:].groupby(yrs).first().loc[sorted(full)]
    infl = cpi.reindex(sim.index).ffill()
    real = sim['value'] / (infl / infl.iloc[0])
    chg = inc.pct_change()
    return pd.Series({
        'Avg. income p.a. (EUR)': inc.mean(),
        'Avg. payout yield': (inc / start_val).mean(),
        'Weakest income year (EUR)': inc.min(),
        'Largest yearly income drop': chg.min(),
        'Years with income drop > 10%': int((chg < -0.10).sum()),
        'Total paid out (EUR)': s['payout'].sum(),
        'End value, nominal (EUR)': sim['value'].iloc[-1],
        'End value, real (x start)': real.iloc[-1] / w0,
        'Total costs (EUR)': sim['cost'].sum(),
    }), inc, real


# =====================================================================
# 5) Planning tables (prospectus sections 2 and 3.2) from stated assumptions
# =====================================================================
SLEEVES = ['EZ', 'US', 'GOV', 'CORP', 'GOLD']


def mix_for_equity(e, gold=0.05, gov_corp=(4, 3)):
    """Equity split 50/50, bonds split 4:3 government/corporate (rounded to whole %), gold fixed."""
    bonds = 1 - e - gold
    gov = round(bonds * gov_corp[0] / sum(gov_corp) * 100) / 100
    return pd.Series({'EZ': e / 2, 'US': e / 2, 'GOV': gov, 'CORP': round((bonds - gov) * 100) / 100, 'GOLD': gold})


def planning_tables(A, equity_shares=(0.4, 0.5, 0.6, 0.7, 0.8), z=1.645):
    src = pd.DataFrame(A['sources']).T
    src['Capital growth after payout'] = src['Expected return'] - src['Income']
    src['Real capital growth'] = src['Capital growth after payout'] - A['inflation']
    mu = pd.Series(A['exp_return'])[SLEEVES]
    inc = pd.Series(A['income'])[SLEEVES]
    vol = pd.Series(A['vol'])[SLEEVES]
    C = pd.DataFrame(A['corr'], index=SLEEVES, columns=SLEEVES)
    S = np.outer(vol, vol) * C.values
    rows = {}
    for e in equity_shares:
        w = mix_for_equity(e)
        er = float(w @ mu) - A['costs']
        v = float(np.sqrt(w.values @ S @ w.values))
        rows[f'{e:.0%}'] = {'Gov / corp bonds': f"{w['GOV']:.0%} / {w['CORP']:.0%}", 'Expected return': er,
                            'Income': float(w @ inc), 'Real capital growth': er - float(w @ inc) - A['inflation'],
                            'Volatility': v, '1-in-20 year': er - z * v}
    eq = pd.DataFrame(rows).T
    eq.index.name = 'Equity share'
    return src, eq

In [ ]:
!pip -q install yfinance

import importlib, json, time, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import evergreen_core as dc
importlib.reload(dc)

pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.width', 200)
NAVY, GOLD, GREY, TEAL, RED, SAND = '#0B3C5D', '#C9A227', '#9AA5B1', '#328CC1', '#B23A48', '#E2B55E'
plt.rcParams.update({'figure.figsize': (11, 4.5), 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3})
PCT = mtick.PercentFormatter(1.0)

## 2. Parameters
Every parameter below is taken from the strategy paper. Change them here only; all later cells read them.

In [ ]:
# ---------- sample (set explicitly) ----------
START_DATA = '2008-01-01'                 # stock dividends from 2009 feed the first five-year screen
START      = pd.Period('2013-12', 'M')    # first selection: last trading day of December 2013
END        = pd.Period('2026-10', 'M')    # last complete month -> monthly returns Jan 2014 - Oct 2026

# ---------- client ----------
W0 = 20_000_000                           # EUR 20 m of liquid sale proceeds (illustrative)

# ---------- strategic weights (paper, section 3) ----------
WEIGHTS = {'EZ': 0.30, 'US': 0.30, 'GOV': 0.20, 'CORP': 0.15, 'GOLD': 0.05}
GROUP   = {'EZ': 'EQ', 'US': 'EQ', 'GOV': 'BOND', 'CORP': 'BOND', 'GOLD': 'GOLD', 'LINK': 'BOND'}

# ---------- Eurozone stock selection rule (paper, section 4) ----------
RULE = dict(n_stocks=20,          # 20 stocks, 5% of the sleeve each
            hist_years=5,         # five-year dividend window (calendar years, ex-date)
            min_paid_years=5,     # dividend paid in every year of the window
            cut_tol=0.10,         # never cut by more than 10% from one year to the next
            net_growth=True,      # latest year >= first year of the window
            max_yield=0.10,       # trailing yield above 10% -> yield trap / data error
            loser_quantile=0.05,  # drop the worst 5% 12-month price performers (if negative)
            buffer_rank=30,       # holdings ranked 21-30 that still pass stay
            max_per_sector=4, max_per_country=6,
            fallback='EXW1.DE')   # empty slots and takeover proceeds -> EURO STOXX 50 ETF

# ---------- rebalancing, payout and costs (paper, section 5) ----------
CFG = dict(w0=W0,
           rebal_month=12,                                    # reselect + reset in December
           band_months=(3, 6, 9),                             # band check at the other quarter-ends
           bands={'EQ': 0.05, 'BOND': 0.05, 'GOLD': 0.02},    # 55-65% / 30-40% / 3-7%
           payout='income',                                   # pay out every dividend and coupon
           payout_rate=0.035, smooth_months=36,               # only for the 'smoothed' alternative
           tc_stock=0.0010, tc_etf=0.0005, cost_mult=1.0,     # + FR/IT/ES transaction taxes on purchases
           fallback=RULE['fallback'])

# ---------- instruments (Xetra, EUR); first ticker with complete data is used ----------
INSTR = {'US':     ['SPYD.DE'],               # SPDR S&P U.S. Dividend Aristocrats UCITS (Dist)
         'GOV':    ['EUNH.DE'],               # iShares Core EUR Govt Bond (Dist)
         'CORP':   ['EUN5.DE', 'IEAC.AS'],    # iShares Core EUR Corp Bond (Dist)
         'GOLD':   ['4GLD.DE'],               # Xetra-Gold
         'EZ_ETF': ['EXW1.DE'],               # iShares Core EURO STOXX 50 (DE): B1 and empty slots
         'SP500':  ['IUSA.DE', 'VUSA.DE'],    # S&P 500 (Dist): B1
         'WORLD':  ['EUNL.DE'],               # iShares Core MSCI World (Acc): B2
         'DIV30':  ['EXSG.DE'],               # iShares EURO STOXX Select Dividend 30: B3
         'LINK':   ['IBCI.DE']}               # EUR inflation-linked govt bonds (app option)
NEEDS_DIST = {'US', 'GOV', 'CORP', 'EZ_ETF', 'SP500', 'DIV30', 'LINK'}
US_PROXY, FX = 'SDY', 'EURUSD=X'              # SDY tracks the same index (USD), used only if SPYD.DE lacks distributions
FRED = {'rf': 'IR3TIB01EZM156N', 'hicp': 'CP0000EZ19M086NEST', 'us_rate': 'TB3MS'}

CACHE = Path('/content/evergreen_cache') if Path('/content').exists() else Path('evergreen_cache')
CACHE.mkdir(exist_ok=True)
print(f'Backtest: selection {START}, returns {START + 1} -> {END} ({len(pd.period_range(START + 1, END, freq="M"))} months)')

## 3. Planning assumptions (prospectus sections 2 and 3.2)
These are **assumptions, not forecasts**; sources are listed in the paper. The notebook recomputes both tables so every prospectus number traces back to code.
The paper gives volatilities and correlation ranges ("0.65 between the equity sleeves, 0.1–0.4 between equities and bonds"); the full matrix below is a reconstruction that reproduces the paper's volatility column exactly. **The group should confirm it.**

In [ ]:
ASSUMPTIONS = dict(
    exp_return={'EZ': 0.065, 'US': 0.060, 'GOV': 0.039, 'CORP': 0.035, 'GOLD': 0.020},
    income    ={'EZ': 0.045, 'US': 0.028, 'GOV': 0.027, 'CORP': 0.030, 'GOLD': 0.000},
    vol       ={'EZ': 0.17,  'US': 0.15,  'GOV': 0.05,  'CORP': 0.045, 'GOLD': 0.15},
    #            EZ    US   GOV  CORP  GOLD
    corr=[[1.00, 0.65, 0.10, 0.40, 0.10],
          [0.65, 1.00, 0.10, 0.40, 0.10],
          [0.10, 0.10, 1.00, 0.90, 0.10],
          [0.40, 0.40, 0.90, 1.00, 0.10],
          [0.10, 0.10, 0.10, 0.10, 1.00]],
    inflation=0.02, costs=0.0015,
    sources={'Eurozone dividend stocks':   {'Expected return': 0.065, 'Income': 0.045},
             'US dividend growers':        {'Expected return': 0.060, 'Income': 0.028},
             'Euro government bonds':      {'Expected return': 0.039, 'Income': 0.027},
             'Euro corporate bonds':       {'Expected return': 0.035, 'Income': 0.030},
             'Euro high yield (not held)': {'Expected return': 0.046, 'Income': 0.058},
             'Gold':                       {'Expected return': 0.020, 'Income': 0.000}})

src_tbl, eq_tbl = dc.planning_tables(ASSUMPTIONS)
print('Does each income source pay for itself? (paper, section 2)')
display(src_tbl.style.format('{:+.1%}'))
print('Choosing the equity share (paper, section 3.2) - net of 0.15% costs, normal approximation for the 1-in-20 year')
display(eq_tbl.style.format({c: '{:.1%}' for c in eq_tbl.columns if c != 'Gov / corp bonds'}))
w_base = pd.Series(WEIGHTS)
inc_base = (w_base * pd.Series(ASSUMPTIONS['income'])).sum()
print(f"Base mix: expected income {inc_base:.2%}; largest single source "
      f"{(w_base * pd.Series(ASSUMPTIONS['income'])).max() / inc_base:.0%} of the payout")

## 4. Universe and data
**Universe.** All current MSCI EMU constituents (iShares Core MSCI EMU holdings, 25 Nov 2026), mapped to Yahoo tickers: 211 stocks. Two constituents listed only in New York (AerCap, Nebius) are left out. The list is stored as a CSV in the group's public GitHub repository and loaded at run time (allowed by the instructor on the Moodle forum, 26 Nov 2026). If the repository cannot be reached, the notebook falls back to the 155-stock Eurozone list of the first draft.

Sectors and countries are today's classification from the index provider, not point-in-time.

In [ ]:
GITHUB_RAW = 'https://raw.githubusercontent.com/NiklasWMueller2404/evergreen-family-income/main'   # <- set the account name once
UNIVERSE_CSV_URL = f'{GITHUB_RAW}/data/universe_msci_emu.csv'

# Fallback: 155-stock Eurozone list of the first draft

UNIVERSE = [
 # Germany
 ('SAP.DE','SAP','Technology'),('SIE.DE','Siemens','Industrials'),('ALV.DE','Allianz','Financials'),
 ('MUV2.DE','Munich Re','Financials'),('DTE.DE','Deutsche Telekom','Communication'),('BAS.DE','BASF','Materials'),
 ('BAYN.DE','Bayer','Health Care'),('BMW.DE','BMW','Consumer Discretionary'),('MBG.DE','Mercedes-Benz','Consumer Discretionary'),
 ('VOW3.DE','Volkswagen Vz','Consumer Discretionary'),('DHL.DE','DHL Group','Industrials'),('EOAN.DE','E.ON','Utilities'),
 ('RWE.DE','RWE','Utilities'),('IFX.DE','Infineon','Technology'),('ADS.DE','adidas','Consumer Discretionary'),
 ('DB1.DE','Deutsche Börse','Financials'),('HEN3.DE','Henkel Vz','Consumer Staples'),('BEI.DE','Beiersdorf','Consumer Staples'),
 ('MRK.DE','Merck KGaA','Health Care'),('FRE.DE','Fresenius','Health Care'),('HNR1.DE','Hannover Rück','Financials'),
 ('DBK.DE','Deutsche Bank','Financials'),('CBK.DE','Commerzbank','Financials'),('HEI.DE','Heidelberg Materials','Materials'),
 ('CON.DE','Continental','Consumer Discretionary'),('SY1.DE','Symrise','Materials'),('MTX.DE','MTU Aero Engines','Industrials'),
 ('VNA.DE','Vonovia','Real Estate'),('BNR.DE','Brenntag','Industrials'),('TLX.DE','Talanx','Financials'),
 ('FNTN.DE','freenet','Communication'),('EVK.DE','Evonik','Materials'),('LEG.DE','LEG Immobilien','Real Estate'),
 # France
 ('MC.PA','LVMH','Consumer Discretionary'),('OR.PA',"L'Oréal",'Consumer Staples'),('TTE.PA','TotalEnergies','Energy'),
 ('SAN.PA','Sanofi','Health Care'),('AI.PA','Air Liquide','Materials'),('SU.PA','Schneider Electric','Industrials'),
 ('BNP.PA','BNP Paribas','Financials'),('CS.PA','AXA','Financials'),('DG.PA','Vinci','Industrials'),
 ('EL.PA','EssilorLuxottica','Health Care'),('KER.PA','Kering','Consumer Discretionary'),('RI.PA','Pernod Ricard','Consumer Staples'),
 ('BN.PA','Danone','Consumer Staples'),('ENGI.PA','Engie','Utilities'),('ORA.PA','Orange','Communication'),
 ('GLE.PA','Société Générale','Financials'),('ACA.PA','Crédit Agricole','Financials'),('VIE.PA','Veolia','Utilities'),
 ('SGO.PA','Saint-Gobain','Industrials'),('CAP.PA','Capgemini','Technology'),('AIR.PA','Airbus','Industrials'),
 ('SAF.PA','Safran','Industrials'),('HO.PA','Thales','Industrials'),('PUB.PA','Publicis','Communication'),
 ('CA.PA','Carrefour','Consumer Staples'),('ML.PA','Michelin','Consumer Discretionary'),('RNO.PA','Renault','Consumer Discretionary'),
 ('EN.PA','Bouygues','Industrials'),('SW.PA','Sodexo','Consumer Discretionary'),('LR.PA','Legrand','Industrials'),
 ('SCR.PA','SCOR','Financials'),('COV.PA','Covivio','Real Estate'),('GET.PA','Getlink','Industrials'),
 ('RUI.PA','Rubis','Utilities'),('TEP.PA','Teleperformance','Industrials'),
 # Italy
 ('ENEL.MI','Enel','Utilities'),('ENI.MI','Eni','Energy'),('ISP.MI','Intesa Sanpaolo','Financials'),
 ('UCG.MI','UniCredit','Financials'),('G.MI','Generali','Financials'),('SRG.MI','Snam','Utilities'),
 ('TRN.MI','Terna','Utilities'),('PST.MI','Poste Italiane','Financials'),('MB.MI','Mediobanca','Financials'),
 ('TIT.MI','Telecom Italia','Communication'),('A2A.MI','A2A','Utilities'),('HER.MI','Hera','Utilities'),
 ('TEN.MI','Tenaris','Energy'),('PRY.MI','Prysmian','Industrials'),('LDO.MI','Leonardo','Industrials'),
 ('BAMI.MI','Banco BPM','Financials'),('IG.MI','Italgas','Utilities'),('AZM.MI','Azimut','Financials'),
 ('BMED.MI','Banca Mediolanum','Financials'),('REC.MI','Recordati','Health Care'),
 # Spain
 ('SAN.MC','Banco Santander','Financials'),('BBVA.MC','BBVA','Financials'),('IBE.MC','Iberdrola','Utilities'),
 ('ITX.MC','Inditex','Consumer Discretionary'),('TEF.MC','Telefónica','Communication'),('REP.MC','Repsol','Energy'),
 ('ENG.MC','Enagás','Utilities'),('ELE.MC','Endesa','Utilities'),('NTGY.MC','Naturgy','Utilities'),
 ('RED.MC','Redeia','Utilities'),('ACS.MC','ACS','Industrials'),('AENA.MC','Aena','Industrials'),
 ('CABK.MC','CaixaBank','Financials'),('MAP.MC','Mapfre','Financials'),('AMS.MC','Amadeus','Technology'),
 ('GRF.MC','Grifols','Health Care'),('BKT.MC','Bankinter','Financials'),('FER.MC','Ferrovial','Industrials'),
 # Netherlands
 ('ASML.AS','ASML','Technology'),('INGA.AS','ING','Financials'),('AD.AS','Ahold Delhaize','Consumer Staples'),
 ('PHIA.AS','Philips','Health Care'),('HEIA.AS','Heineken','Consumer Staples'),('NN.AS','NN Group','Financials'),
 ('KPN.AS','KPN','Communication'),('WKL.AS','Wolters Kluwer','Industrials'),('RAND.AS','Randstad','Industrials'),
 ('AKZA.AS','Akzo Nobel','Materials'),('ASRNL.AS','ASR Nederland','Financials'),('AGN.AS','Aegon','Financials'),
 ('ABN.AS','ABN AMRO','Financials'),('SBMO.AS','SBM Offshore','Energy'),
 # Belgium
 ('ABI.BR','AB InBev','Consumer Staples'),('KBC.BR','KBC','Financials'),('UCB.BR','UCB','Health Care'),
 ('ACKB.BR','Ackermans & van Haaren','Financials'),('PROX.BR','Proximus','Communication'),('AGS.BR','Ageas','Financials'),
 ('ELI.BR','Elia','Utilities'),('COFB.BR','Cofinimmo','Real Estate'),('GBLB.BR','GBL','Financials'),
 # Finland
 ('NOKIA.HE','Nokia','Technology'),('SAMPO.HE','Sampo','Financials'),('FORTUM.HE','Fortum','Utilities'),
 ('UPM.HE','UPM-Kymmene','Materials'),('STERV.HE','Stora Enso R','Materials'),('KNEBV.HE','Kone','Industrials'),
 ('NESTE.HE','Neste','Energy'),('ELISA.HE','Elisa','Communication'),('WRT1V.HE','Wärtsilä','Industrials'),
 ('NDA-FI.HE','Nordea','Financials'),('ORNBV.HE','Orion','Health Care'),('KESKOB.HE','Kesko B','Consumer Staples'),
 # Austria
 ('OMV.VI','OMV','Energy'),('VER.VI','Verbund','Utilities'),('EBS.VI','Erste Group','Financials'),
 ('VIG.VI','Vienna Insurance Group','Financials'),('POST.VI','Österreichische Post','Industrials'),
 ('TKA.VI','Telekom Austria','Communication'),('ANDR.VI','Andritz','Industrials'),('RBI.VI','Raiffeisen Bank Intl.','Financials'),
 ('WIE.VI','Wienerberger','Materials'),('BG.VI','BAWAG','Financials'),
 # Portugal
 ('EDP.LS','EDP','Utilities'),('GALP.LS','Galp','Energy'),('JMT.LS','Jerónimo Martins','Consumer Staples'),
 ('NOS.LS','NOS','Communication'),('NVG.LS','Navigator','Materials'),('REN.LS','REN','Utilities'),
]

SUFFIX = {'.DE': 'Germany', '.PA': 'France', '.MI': 'Italy', '.MC': 'Spain', '.AS': 'Netherlands',
          '.BR': 'Belgium', '.HE': 'Finland', '.VI': 'Austria', '.LS': 'Portugal', '.IR': 'Ireland'}
try:
    meta = pd.read_csv(UNIVERSE_CSV_URL).set_index('ticker')
    UNIVERSE_SOURCE = 'MSCI EMU constituents (GitHub CSV)'
except Exception as e:
    print('GitHub universe not reachable -> fallback list. Reason:', str(e)[:80])
    meta = pd.DataFrame(UNIVERSE, columns=['ticker', 'name', 'sector']).set_index('ticker')
    UNIVERSE_SOURCE = 'first-draft 155-stock list (fallback)'
if 'country' not in meta.columns:
    meta['country'] = [SUFFIX['.' + t.split('.')[-1]] for t in meta.index]
meta = meta[['name', 'sector', 'country']]
print(f'{len(meta)} stocks in the universe: {UNIVERSE_SOURCE}')
display(meta.groupby('country').size().to_frame('stocks').T)

In [ ]:
def cached(name, fn):
    path = CACHE / f'{name}_{END}.pkl'
    if path.exists():
        return pd.read_pickle(path)
    obj = fn()
    pd.to_pickle(obj, path)
    return obj

t0 = time.time()
ALL = list(meta.index) + sorted({t for c in INSTR.values() for t in c}) + [US_PROXY, FX]
close_raw, div_m = cached('prices', lambda: dc.download_prices(ALL, START_DATA, END))
full_idx = pd.period_range(close_raw.index.min(), END, freq='M')
close_raw, div_m = close_raw.reindex(full_idx), div_m.reindex(full_idx).fillna(0.0)
print(f'Prices: {close_raw.shape[1]} tickers, {close_raw.index.min()} -> {close_raw.index.max()} ({time.time() - t0:.0f}s)')

failed = [t for t in ALL if t not in close_raw.columns]
print(f'Failed downloads: {len(failed)} {failed}' if failed else 'All tickers downloaded')
meta = meta.loc[meta.index.intersection(close_raw.columns)]

# ---------- pick instruments: complete prices and (for distributing funds) a distribution every full year ----------
def usable(tk, need_dist):
    if tk not in close_raw.columns or close_raw[tk].loc[START:END].notna().mean() < 0.95:
        return False
    if need_dist:
        d = div_m[tk].loc[START + 1:END]
        yearly = d.groupby(d.index.year).sum()
        full_years = [y for y in yearly.index if y < END.year]
        return bool((yearly.loc[full_years] > 0).all())
    return True

eurusd = close_raw[FX].ffill()
T, notes = {}, []
for k, cands in INSTR.items():
    ok = [c for c in cands if usable(c, k in NEEDS_DIST)]
    T[k] = ok[0] if ok else None
if T['US'] is None and US_PROXY in close_raw.columns:      # SPYD.DE without distributions -> SDY in EUR
    close_raw['SDY_EUR'], div_m['SDY_EUR'] = dc.usd_to_eur(close_raw[US_PROXY], div_m[US_PROXY], eurusd)
    T['US'] = 'SDY_EUR'
    notes.append('US sleeve: SPYD.DE distributions incomplete on Yahoo -> SDY (same index) converted to EUR')
for k in ['US', 'GOV', 'CORP', 'GOLD', 'EZ_ETF', 'SP500', 'WORLD', 'DIV30']:
    assert T[k] is not None, f'{k}: no usable ticker in {INSTR[k]} - check Yahoo symbols'
print('Instruments used:', T)
for n in notes: print('Note:', n)

# ---------- FRED: risk-free rate, euro inflation, US rate (hedge option) ----------
rate, hicp, us_rate = (cached(f'fred_{v}', lambda v=v: dc.load_fred(v)) for v in FRED.values())
print(f'FRED: 3M rate to {rate.index.max()}, HICP to {hicp.index.max()}, US T-bill to {us_rate.index.max()} (later months carried forward)')
r_eur_ann = (rate.reindex(full_idx).ffill() / 100).shift(1)    # known at the start of the month
r_usd_ann = (us_rate.reindex(full_idx).ffill() / 100).shift(1)
rf = r_eur_ann / 12
cpi = hicp.reindex(full_idx).ffill()

# ---------- synthetic hedged US sleeve (app / sensitivity only) ----------
close_raw['US_HEDGED'], div_m['US_HEDGED'] = dc.fx_hedged_series(close_raw[T['US']], div_m[T['US']], eurusd, r_eur_ann, r_usd_ann)

PX = close_raw.ffill()                                         # prices for valuation
LAST_VALID = {k: close_raw[k].last_valid_index() for k in meta.index}
ETFS = {v for v in T.values() if v} | {'US_HEDGED', 'SDY_EUR', RULE['fallback']}
CFG.update(etfs=ETFS, country=meta['country'].to_dict())

## 5. Data checks
Before any number goes into the prospectus: distributions present every year, ETF calendar-year returns close to the issuer factsheets, and plausible dividend histories.

In [ ]:
check = [T[k] for k in ['US', 'GOV', 'CORP', 'EZ_ETF', 'SP500', 'DIV30'] if T[k]]
dist_yield = pd.DataFrame({k: (div_m[k] / PX[k]).groupby(div_m.index.year).sum() for k in check}).loc[START.year:]
print('Distribution yield per calendar year (0 = missing distributions on Yahoo):')
display(dist_yield.style.format('{:.2%}'))

tr = pd.DataFrame({k: dc.total_return(PX[k], div_m[k]) for k in check + [T['GOLD'], T['WORLD']]}).loc[START + 1:END]
cal_etf = (1 + tr).groupby(tr.index.year).prod() - 1
print('Calendar-year total returns in EUR (compare with the issuer factsheets):')
display(cal_etf.style.format('{:+.1%}'))

FACTSHEET = {T['CORP']: {2022: -0.139, 2023: 0.080, 2024: 0.046, 2025: 0.031}}   # paper, section 8
for tk, ref in FACTSHEET.items():
    gap = pd.Series({y: cal_etf.at[y, tk] - v for y, v in ref.items() if y in cal_etf.index})
    print(f'{tk}: gap to factsheet (pp):', (gap * 100).round(2).to_dict(),
          '-> OK' if (gap.abs() < 0.005).all() else '-> CHECK: gap > 0.5 pp usually means missing distributions')

stocks = meta.index
annual_all = div_m[stocks].groupby(div_m.index.year).sum()
first_px = close_raw[stocks].apply(lambda c: c.first_valid_index())
print(f'\nStocks with prices before 2009: {(first_px < pd.Period("2009-01", "M")).sum()} of {len(stocks)}')
print(f'Stocks without any dividend in the data: {(annual_all.sum() == 0).sum()}')
gone = {k: v for k, v in LAST_VALID.items() if v is not None and v < END}
print(f'Stocks whose prices stop before {END} (takeover / delisting / symbol change): {len(gone)}', gone if gone else '')

## 6. Eurozone stock selection
On the last trading day of each December, using only prices and dividends known on that date:
1. **Dividend record:** dividend paid in each of the last five calendar years, never cut by more than 10% year on year, latest year ≥ first year.
2. **Yield trap filter:** trailing 12-month yield ≤ 10%; drop the worst 5% 12-month price performers if their return is negative.
3. **Rank** by trailing 12-month yield.
4. **Select with caps and buffer:** holdings that still pass and rank in the top 30 stay; fill to 20 in rank order, max. 4 per sector and 6 per country.
5. **Equal weights** (5% of the sleeve, 1.5% of the portfolio); empty places go to the EURO STOXX 50 ETF.

In [ ]:
REBAL_DATES = [p for p in pd.period_range(START, END, freq='M') if p.month == CFG['rebal_month']]

def selection_for(rule):
    return dc.run_selection(REBAL_DATES, close_raw, div_m, meta, rule)

SEL = selection_for(RULE)
ez_rule = lambda t: SEL[t][0]

counts = pd.DataFrame({t.year: c for t, (w, i, c) in SEL.items()}).T
counts.index.name = 'selection (Dec)'
print('How many stocks pass each screen:')
display(counts)

# checks: weights, caps, no stale data
for t, (w, info, c) in SEL.items():
    assert abs(w.sum() - 1) < 1e-9, t
    assert info['sector'].value_counts().max() <= RULE['max_per_sector'], t
    assert info['country'].value_counts().max() <= RULE['max_per_country'], t
print('Checks passed: weights sum to 1, sector and country caps hold, selection uses data up to t only')

flag = pd.concat({t.year: i[i['yield_ttm'] > 0.08] for t, (w, i, c) in SEL.items()})
print(f'\nSelected stocks with a trailing yield above 8% (check for special dividends): {len(flag)}')
if len(flag): display(flag[['name', 'yield_ttm']].style.format({'yield_ttm': '{:.1%}'}))

last = REBAL_DATES[-1]
print(f'\nHoldings selected in {last}:')
display(SEL[last][1].sort_values('yield_ttm', ascending=False).style.format({'yield_ttm': '{:.2%}'}))
turnover = pd.Series({t.year: 1 - len(set(SEL[t][1].index) & set(SEL[p][1].index)) / RULE['n_stocks']
                      for p, t in zip(REBAL_DATES[:-1], REBAL_DATES[1:])})
print('Share of the 20 places replaced each year:', turnover.round(2).to_dict())

sec = pd.DataFrame({t.year: w.groupby(meta['sector'].reindex(w.index).fillna('EURO STOXX 50 ETF (empty slots)')).sum()
                    for t, (w, i, c) in SEL.items()}).T.fillna(0)
ax = sec.plot(kind='bar', stacked=True, figsize=(11, 4.2), colormap='tab20', width=0.85)
ax.yaxis.set_major_formatter(PCT)
ax.set_title('Sector weights inside the Eurozone sleeve at each December selection', loc='left', fontweight='bold')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, frameon=False); plt.tight_layout(); plt.show()

## 7. Backtest vs. benchmarks
All portfolios follow the same rebalancing (December reset, quarterly bands), payout rule and cost model. Returns are time-weighted, so payouts do not distort them.

| Portfolio | Composition | Question it answers |
|---|---|---|
| **Evergreen** | 30% our rule + 30% SPYD + 20% EUNH + 15% EUN5 + 5% gold | – |
| **B1 Policy twin (main)** | 30% EURO STOXX 50 + 30% S&P 500 + 20% EUNH + 15% EUN5 + 5% gold | Does tilting equities to income help, at the same asset mix? |
| **B2 Classic 60/40** | 60% MSCI World + 40% EUNH | Would a standard balanced mandate have served the family better? |
| **B3 Dividend-ETF variant** | B1 with EURO STOXX Select Dividend 30 as Eurozone sleeve | Does our stock rule beat an off-the-shelf dividend fund? |
| 1/N diagnostic | Evergreen with all universe stocks equally weighted in the Eurozone sleeve | How much comes from the screens, how much from the universe? |

Risk-free rate: euro area 3-month rate (FRED). Beta is not annualised.

In [ ]:
def fixed(tk):
    return lambda t: pd.Series({tk: 1.0})

def sleeve(w, group, fn):
    return {'weight': w, 'group': group, 'fn': fn, 'fallback': RULE['fallback']}

def five_sleeves(W, ez_fn, us_tk, gov_tk=None, link_tk=None):
    spec = {'EZ':   sleeve(W['EZ'], 'EQ', ez_fn),
            'US':   sleeve(W['US'], 'EQ', fixed(us_tk)),
            'GOV':  sleeve(W['GOV'], 'BOND', fixed(gov_tk or T['GOV'])),
            'CORP': sleeve(W['CORP'], 'BOND', fixed(T['CORP'])),
            'GOLD': sleeve(W['GOLD'], 'GOLD', fixed(T['GOLD']))}
    if W.get('LINK', 0) > 0:
        spec['LINK'] = sleeve(W['LINK'], 'BOND', fixed(link_tk or T['LINK']))
    return spec

ew_universe = lambda t: dc.equal_weight_universe(t, close_raw, meta)
SPECS = {
    'Evergreen':      five_sleeves(WEIGHTS, ez_rule, T['US']),
    'B1 Policy twin': five_sleeves(WEIGHTS, fixed(T['EZ_ETF']), T['SP500']),
    'B2 60/40':       {'WORLD': sleeve(0.60, 'EQ', fixed(T['WORLD'])), 'GOV': sleeve(0.40, 'BOND', fixed(T['GOV']))},
    'B3 Dividend ETF': five_sleeves(WEIGHTS, fixed(T['DIV30']), T['SP500']),
    '1/N diagnostic': five_sleeves(WEIGHTS, ew_universe, T['US']),
}
COLORS = {'Evergreen': NAVY, 'B1 Policy twin': SAND, 'B2 60/40': GREY, 'B3 Dividend ETF': TEAL, '1/N diagnostic': RED}

def run(spec, cfg=CFG):
    return dc.simulate(spec, PX, div_m, START, END, cfg, LAST_VALID)

t0 = time.time()
SIMS = {k: run(s) for k, s in SPECS.items()}
R = pd.DataFrame({k: s['ret'] for k, s in SIMS.items()}).dropna()
print(f'Simulated {len(SIMS)} portfolios, {len(R)} months ({time.time() - t0:.0f}s)')

# checks: after every December reset the groups sit exactly on target
ev = SIMS['Evergreen']
tgt = pd.Series(WEIGHTS).groupby(GROUP).sum()
for g in tgt.index:
    dev = (ev.loc[ev['event'] == 'annual', f'w_{g}'] - tgt[g]).abs().max()
    assert dev < 1e-6, (g, dev)
print('Check passed: group weights back on target after every December reset')
print('Band resets (quarter-ends):', [str(p) for p in ev.index[ev['event'] == 'band']] or 'none')

fig, ax = plt.subplots()
for k in R:
    c = pd.concat([pd.Series([1.0], index=[R.index[0] - 1]), (1 + R[k]).cumprod()])
    ax.plot(c.index.to_timestamp(), c.values, label=k, color=COLORS[k], lw=2.6 if k == 'Evergreen' else 1.5,
            ls=':' if k == '1/N diagnostic' else '-')
ax.axhline(1, color='k', lw=0.6)
ax.set_title(f'Cumulative total return, net of costs (start = 1), {START + 1} to {END}', loc='left', fontweight='bold')
ax.legend(frameon=False); plt.show()

In [ ]:
PCT_ROWS = ['Ann. average return', 'CAGR', 'Volatility', 'Max drawdown', 'Worst 12-month return',
            'Alpha (ann.)', 'Tracking error']
def fmt(df):
    rows = [r for r in df.index if r in PCT_ROWS]
    return df.style.format('{:.2f}').format('{:.2%}', subset=pd.IndexSlice[rows, :])

abs_tbl = pd.DataFrame({k: dc.standalone_stats(R[k], rf) for k in R})
print('Absolute measures (monthly data, annualised):')
display(fmt(abs_tbl))

BENCH = ['B1 Policy twin', 'B2 60/40', 'B3 Dividend ETF', '1/N diagnostic']
rel_tbl = pd.DataFrame({f'vs {b}': dc.relative_stats(R['Evergreen'], R[b], rf) for b in BENCH})
print('Evergreen relative to each benchmark (alpha, beta, IR depend on the benchmark):')
display(fmt(rel_tbl))

## 8. Where does the difference come from? Sleeve attribution
Evergreen and B1 share their bonds and gold; they differ in **two** sleeves. Each sleeve is tested on its own (100% in that sleeve, same rules), so the effect of our stock rule is not mixed up with the choice of the US fund.

In [ ]:
SLEEVE_TESTS = {'EZ: our rule': ez_rule, 'EZ: EURO STOXX 50': fixed(T['EZ_ETF']),
                'EZ: Select Dividend 30': fixed(T['DIV30']), 'EZ: 1/N universe': ew_universe,
                'US: SPYD (dividend aristocrats)': fixed(T['US']), 'US: S&P 500': fixed(T['SP500'])}
RS = pd.DataFrame({k: run({'X': sleeve(1.0, 'EQ', f)})['ret'] for k, f in SLEEVE_TESTS.items()}).dropna()
pairs = [('EZ: our rule', 'EZ: EURO STOXX 50'), ('EZ: our rule', 'EZ: Select Dividend 30'),
         ('EZ: our rule', 'EZ: 1/N universe'), ('US: SPYD (dividend aristocrats)', 'US: S&P 500')]
att = pd.DataFrame({f'{a.split(": ")[1]} vs {b.split(": ")[1]}':
                    pd.concat([dc.standalone_stats(RS[a], rf)[['CAGR', 'Volatility', 'Sharpe ratio']],
                               pd.Series({'Benchmark CAGR': dc.standalone_stats(RS[b], rf)['CAGR']}),
                               dc.relative_stats(RS[a], RS[b], rf)]) for a, b in pairs})
PCT_ROWS.append('Benchmark CAGR')
display(fmt(att))

## 9. Client view: can the family live on it, and does the capital survive?
Income in euros, its stability, and **real** wealth after payouts (deflated by euro area HICP). The risk budget in the paper is a 1-in-20 bad year of about −10%; the worst rolling 12-month return shows how the backtest compares.

In [ ]:
CLIENT, INCOME, REAL = {}, {}, {}
for k in ['Evergreen', 'B1 Policy twin', 'B2 60/40', 'B3 Dividend ETF']:
    CLIENT[k], INCOME[k], REAL[k] = dc.client_stats(SIMS[k], cpi, W0)
client_tbl = pd.DataFrame(CLIENT)
client_tbl.loc['Worst 12-month return'] = abs_tbl.loc['Worst 12-month return', client_tbl.columns]
eur_rows = [r for r in client_tbl.index if '(EUR)' in r]
show = client_tbl.astype(object)
for r in client_tbl.index:
    show.loc[r] = [f'{v:,.0f}' if r in eur_rows else (f'{v:.2f}x' if '(x' in r else
                   (f'{v:.0f}' if r.startswith('Years') else f'{v:.2%}')) for v in client_tbl.loc[r]]
display(show)
worst = abs_tbl.at['Worst 12-month return', 'Evergreen']
print(f'Risk budget: worst 12-month return {worst:.1%} vs. about -10% planned -> '
      + ('within budget' if worst >= -0.105 else 'BUDGET BREACHED - discuss in the prospectus'))
print('Note: B2 holds an accumulating MSCI World ETF, so its payout and real wealth are not comparable '
      '(it pays out only the bond coupons).')

inc = pd.DataFrame(INCOME)[['Evergreen', 'B1 Policy twin', 'B3 Dividend ETF']] / 1e3
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2))
inc.plot(kind='bar', ax=ax1, color=[COLORS[k] for k in inc.columns], width=0.8)
ax1.set_title(f'Income paid out per year (EUR thousand, on EUR {W0/1e6:.0f} m)', loc='left', fontweight='bold')
ax1.legend(fontsize=8, frameon=False)
for k in ['Evergreen', 'B1 Policy twin', 'B3 Dividend ETF']:
    ax2.plot(REAL[k].index.to_timestamp(), REAL[k].values / W0, color=COLORS[k], lw=2.4 if k == 'Evergreen' else 1.5, label=k)
ax2.axhline(1, color=RED, ls='--', lw=1, label='starting wealth, real')
ax2.set_title('Real wealth after payouts (start = 1)', loc='left', fontweight='bold'); ax2.legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

dd = pd.DataFrame({k: (1 + R[k]).cumprod() / (1 + R[k]).cumprod().cummax() - 1 for k in R if k != '1/N diagnostic'})
ax = dd.plot(color=[COLORS[k] for k in dd], figsize=(11, 3.2), lw=1.4)
ax.yaxis.set_major_formatter(PCT)
ax.set_title('Drawdowns', loc='left', fontweight='bold'); ax.legend(fontsize=8, frameon=False); plt.show()

## 10. Stress years
- **2020:** Covid; the ECB asked banks to suspend dividends, many firms cut.
- **2022:** rate shock; stocks **and** bonds fell together.

In [ ]:
cal = (1 + R).groupby(R.index.year).prod() - 1
inc_chg = pd.DataFrame(INCOME).pct_change()
stress = pd.concat({'Total return': cal.loc[[2020, 2022]],
                    'Income change vs. prior year': inc_chg.loc[[2020, 2021, 2022]]})
display(stress.style.format('{:+.1%}', na_rep='–'))
cal.index = [f'{y} (Jan-{END.strftime("%b")})' if y == END.year and END.month < 12 else str(y) for y in cal.index]
print('Calendar-year returns:')
display(cal.style.format('{:+.1%}').background_gradient(cmap='RdYlGn', axis=None, vmin=-0.2, vmax=0.2))

## 11. Sensitivity
How much do the results depend on single choices? **Not** for picking the best parameters (that would be data snooping), but to show whether the main conclusions hold. Rule variants show what the open group decision on the dividend screen changes.

In [ ]:
def scaled(W, **override):
    out = {**W, **override}
    free = [k for k in W if k not in override]
    rest = 1 - sum(override.values())
    tot = sum(W[k] for k in free)
    return {k: (out[k] if k in override else W[k] / tot * rest) for k in out}

def equity_mix(e):
    return dc.mix_for_equity(e).to_dict()

RULE_VARIANTS = {
    'Rule: first draft (no cut > 5%, no net test)': {**RULE, 'cut_tol': 0.05, 'net_growth': False},
    'Rule: paid 4 of 5 years + 5-year growth >= 0': {**RULE, 'min_paid_years': 4, 'cut_tol': None, 'net_growth': True},
}
SEL_V = {k: selection_for(r) for k, r in RULE_VARIANTS.items()}

VARIANTS = {
    'Base case':                    (SPECS['Evergreen'], CFG),
    'Trading costs x2':             (SPECS['Evergreen'], {**CFG, 'cost_mult': 2.0}),
    'No gold (weights rescaled)':   (five_sleeves(scaled(WEIGHTS, GOLD=0.0), ez_rule, T['US']), CFG),
    'US sleeve hedged to EUR':      (five_sleeves(WEIGHTS, ez_rule, 'US_HEDGED'), CFG),
    'Equity share 50%':             (five_sleeves(equity_mix(0.5), ez_rule, T['US']), CFG),
    'Equity share 70%':             (five_sleeves(equity_mix(0.7), ez_rule, T['US']), CFG),
    'Fixed payout 3.5% of 3-yr avg': (SPECS['Evergreen'], {**CFG, 'payout': 'smoothed'}),
}
if T['LINK']:
    VARIANTS['Half of govt bonds in linkers'] = (five_sleeves({**WEIGHTS, 'GOV': 0.10, 'LINK': 0.10}, ez_rule, T['US']), CFG)
for k, sel in SEL_V.items():
    VARIANTS[k] = (five_sleeves(WEIGHTS, lambda t, sel=sel: sel[t][0], T['US']), CFG)

rows = {}
for k, (spec, cfg) in VARIANTS.items():
    s = SIMS['Evergreen'] if k == 'Base case' else run(spec, cfg)
    r = s['ret'].dropna()
    a, rel = dc.standalone_stats(r, rf), dc.relative_stats(r, R['B1 Policy twin'], rf)
    c, _, _ = dc.client_stats(s, cpi, W0)
    rows[k] = {'CAGR': a['CAGR'], 'Volatility': a['Volatility'], 'Sharpe': a['Sharpe ratio'],
               'Max drawdown': a['Max drawdown'], 'Worst 12m': a['Worst 12-month return'],
               'Alpha vs B1': rel['Alpha (ann.)'], 'IR vs B1': rel['Information ratio'],
               'Avg. payout yield': c['Avg. payout yield'], 'Largest income drop': c['Largest yearly income drop'],
               'Real wealth (x)': c['End value, real (x start)']}
sens = pd.DataFrame(rows).T
display(sens.style.format({c: '{:.2%}' for c in sens.columns if c not in ('Sharpe', 'IR vs B1', 'Real wealth (x)')})
        .format({'Sharpe': '{:.2f}', 'IR vs B1': '{:.2f}', 'Real wealth (x)': '{:.2f}x'}))

elig = pd.DataFrame({'Paper rule': counts['eligible'],
                     **{k: pd.Series({t.year: c['eligible'] for t, (w, i, c) in sel.items()}) for k, sel in SEL_V.items()}})
fb = pd.DataFrame({'Paper rule': counts['fallback_slots'],
                   **{k: pd.Series({t.year: c['fallback_slots'] for t, (w, i, c) in sel.items()}) for k, sel in SEL_V.items()}})
fin = pd.DataFrame({'Paper rule': pd.Series({t.year: (i['sector'] == 'Financials').sum() for t, (w, i, c) in SEL.items()}),
                    **{k: pd.Series({t.year: (i['sector'] == 'Financials').sum() for t, (w, i, c) in sel.items()})
                       for k, sel in SEL_V.items()}})
print('Eligible stocks per December selection:'); display(elig.T)
print('Empty places filled with the EURO STOXX 50 ETF:'); display(fb.T)
print('Financials among the 20 holdings:'); display(fin.T)

## 12. Export for the prospectus and the app
Writes all tables and series to `evergreen_results.zip` (downloads automatically in Colab).

In [ ]:
out = Path('evergreen_results'); out.mkdir(exist_ok=True)
params = {'START': str(START), 'END': str(END), 'W0': W0, 'WEIGHTS': WEIGHTS, 'RULE': RULE,
          'CFG': {k: v for k, v in CFG.items() if k not in ('etfs', 'country')}, 'instruments': T, 'notes': notes,
          'universe_size': len(meta), 'universe_source': UNIVERSE_SOURCE, 'failed_downloads': failed}
(out / 'params.json').write_text(json.dumps(params, indent=1, default=str))
R.to_csv(out / 'monthly_returns.csv')
for k, s in SIMS.items(): s.to_csv(out / ('sim_' + ''.join(ch if ch.isalnum() else '_' for ch in k) + '.csv'))
for name, df in {'planning_sources': src_tbl, 'planning_equity_share': eq_tbl, 'absolute': abs_tbl,
                 'relative': rel_tbl, 'sleeve_attribution': att, 'client': client_tbl, 'calendar_returns': cal,
                 'income_per_year': pd.DataFrame(INCOME), 'sensitivity': sens, 'screen_counts': counts,
                 'eligible_by_rule': elig, 'fallback_by_rule': fb, 'dist_yield_etfs': dist_yield, 'etf_calendar': cal_etf}.items():
    df.to_csv(out / f'{name}.csv')
pd.concat({t.year: i for t, (w, i, c) in SEL.items()}).to_csv(out / 'selections.csv')
with zipfile.ZipFile('evergreen_results.zip', 'w') as z:
    for f in out.iterdir(): z.write(f, f.name)
print('Wrote evergreen_results.zip')
try:
    from google.colab import files
    files.download('evergreen_results.zip')
except Exception:
    pass

## 13. Limitations (each needs a sentence in the prospectus)
- **Survivorship bias:** the universe is today's large caps; firms that failed or were taken over since 2013 are missing. This flatters the rule and the 1/N diagnostic equally; B1–B3 are real ETFs and free of it. The cleanest test of the screens is therefore *rule vs. 1/N*.
- **Hindsight in the asset mix:** weights were set in 2026 and revised after a first backtest (45/30/20/5); gold had a very strong last year. Weights are argued from the client profile, the no-gold variant is shown.
- **Sector data:** today's Yahoo sector, not point-in-time.
- **Data quality:** Yahoo dividend records have gaps, special and scrip dividends; checked in section 5, 10% yield cap.
- **Taxes:** transaction taxes modelled; withholding taxes not modelled, income shown gross.
- **Execution:** trades at the selection-day close.
- **Sample:** 2014–2026 misses the 2008 crisis and the 2011 euro crisis peak; planning assumptions are judgmental.
- **Payout timing:** dividends cluster in spring; the 12-month payout reserve that smooths monthly payments sits outside the mandate and the backtest.